# Create Gordon and Betty Moore Foundation Grant Awards (ORG-LEVEL GRANT PATTERN, Method 5 server-rendered HTML)

The Gordon and Betty Moore Foundation is a major US research funder (Science, Environmental Conservation, Patient Care, and San Francisco Bay Area programs). This ingest covers the foundation's public grants database, one row per grant.

**Method 5 (server-rendered HTML, plain requests).** The scraper (`scripts/local/moore_to_s3.py`) paginates the foundation's grant search at `moore.org/grants?...&currentPage=N` (fully server-rendered, no Cloudflare, no SPA), and enriches each grant from its detail page `moore.org/grant-detail?grantId=...`.

**Awarding body:** Gordon and Betty Moore Foundation - F4320306202 (US, ROR 006wxqw41, DOI 10.13039/100000936)

**Source:** the grant search results table gives, per grant, the `grantId` (e.g. `GBMF14251`), the grant title, the program, the term, the award month/year, and the amount. Each grant's detail page gives the grantee `Organization`.

**Schema choices:**
- One row per grant. `funder_award_id` = the source `grantId` (e.g. `GBMF14251`; stable, unique, source-authoritative).
- `display_name` = the grant's own title.
- `funding_type` = `grant`. `funder_scheme` = the **program** (Science, Environmental Conservation, Patient Care, ...).
- **Amount** = the published `$N` figure (USD) → `amount` (DOUBLE) + `currency` = `USD` where present (> 0); NULL otherwise. **§6.7 NOT waived**, never imputed; any `$0`/blank → NULL.
- `start_date` = NULL — the source gives **month precision** ("May 2026"), so no false day-level precision is asserted; `start_year` is derived from the year. `end_date`/`end_year` = NULL (the source gives a planned `term`, not an end date).
- `lead_investigator` = an **org-only** lead: `given_name`/`family_name` NULL and `affiliation.name` = the grantee organization (from the detail page). `affiliation.country` is NULL — the source exposes no grantee country, so it is never guessed.
- `co_lead_investigator` and `investigators` are NULL (the source names no individuals).
- `description` = NULL — the grant summary is client-rendered (Mustache template), not in the server HTML.
- **One row per `grantId`.** The source lists multiple funding actions under the same grantId (initial grant + later supplements/amendments, some title-less); the scraper collapses each grantId to a single row, keeping the **largest published amount** (a real figure — never a synthesized sum), with the descriptive title and earliest year backfilled from the grant's other actions.

**S3 location:** `s3a://openalex-ingest/awards/moore/moore_grants.parquet`

**Provenance:** `moore_foundation` (verified count=0 on production 2026-06-01).


## Step 1: Create staging table from S3


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.moore_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/moore/moore_grants.parquet`;


In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.moore_raw;


## Step 1.5: Inspect raw + program/year/coverage

Expected: 3,490 grants (one row per grantId, 2001-2026; ~4,461 source list rows collapsed by grantId). funder_award_id / program / amount / start_year / grantee_org ~100%; title ~100% (a few grants carry no title in the listing -> display_name falls back to the grant id).


In [ ]:
%sql
DESCRIBE openalex.awards.moore_raw;


In [ ]:
%sql
SELECT * FROM openalex.awards.moore_raw LIMIT 5;


In [ ]:
%sql
SELECT program, start_year, COUNT(*) AS rows, COUNT(amount) AS has_amount, COUNT(grantee_org) AS has_org
FROM openalex.awards.moore_raw
GROUP BY program, start_year
ORDER BY start_year DESC, rows DESC;


In [ ]:
%sql
SELECT grantee_org, COUNT(*) AS rows, ROUND(SUM(amount),0) AS total_usd
FROM openalex.awards.moore_raw
WHERE grantee_org IS NOT NULL
GROUP BY grantee_org
ORDER BY rows DESC
LIMIT 15;


## Step 1.6: Fail-fast - verify Gordon and Betty Moore Foundation funder row exists

Runbook §2.2.4 mandatory check. Must return exactly 1 row.


In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320306202;


## Step 2: Transform to award schema

`display_name` = the grant's own title (fallback to the grant id). `lead_investigator` is an org-only lead: given/family NULL, `affiliation.name` = grantee org, `affiliation.country` NULL (no source country; never guessed). `start_date` NULL (month precision). `funder_scheme` = program. `co_lead_investigator` / `investigators` are NULL.


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.moore_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320306202  -- Gordon and Betty Moore Foundation
)
SELECT
    abs(xxhash64(CONCAT(
        TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
    ))) % 9000000000 AS id,
    COALESCE(n.title, CONCAT('Moore Foundation grant ', n.funder_award_id)) AS display_name,
    CAST(NULL AS STRING) AS description,
    f.funder_id,
    n.funder_award_id,
    CASE WHEN TRY_CAST(n.amount AS DOUBLE) > 0 THEN TRY_CAST(n.amount AS DOUBLE) ELSE NULL END AS amount,
    CASE WHEN TRY_CAST(n.amount AS DOUBLE) > 0 THEN 'USD' ELSE NULL END AS currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    'grant' AS funding_type,
    n.program AS funder_scheme,
    'moore_foundation' AS provenance,
    CAST(NULL AS DATE) AS start_date,
    CAST(NULL AS DATE) AS end_date,
    TRY_CAST(n.start_year AS INT) AS start_year,
    CAST(NULL AS INT) AS end_year,
    CASE
        WHEN n.grantee_org IS NOT NULL THEN struct(
            CAST(NULL AS STRING) AS given_name,
            CAST(NULL AS STRING) AS family_name,
            CAST(NULL AS STRING) AS orcid,
            CAST(NULL AS DATE) AS role_start,
            struct(
                n.grantee_org AS name,
                CAST(NULL AS STRING) AS country,  -- no grantee country in source; never guessed
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
            ) AS affiliation
        )
        ELSE NULL
    END AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    n.landing_page_url AS landing_page_url,
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT(
               TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
           ))) % 9000000000 AS STRING)) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM openalex.awards.moore_raw n
CROSS JOIN funder_resolved f
WHERE n.funder_award_id IS NOT NULL;


## Step 3: Insert into openalex_awards_raw at priority 161


In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'moore_foundation' AND priority = 161;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    161 AS priority  -- Gordon and Betty Moore Foundation grants
FROM openalex.awards.moore_awards;


## Step 6: Verification

§6.7 amount-coverage check **applies** (NOT waived): Moore publishes a `$N` amount on ~100% of grants. Other checks: display_name / start_year / lead (grantee org) ~100%; description NULL (Mustache-rendered summary not in server HTML); currency = [USD]; start_date NULL (month precision).


In [ ]:
%sql
SELECT COUNT(*) AS total_rows FROM openalex.awards.moore_awards;


In [ ]:
%sql
DESCRIBE openalex.awards.moore_awards;


In [ ]:
%sql
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(amount) AS has_amount,
    COUNT(lead_investigator) AS has_lead,
    COUNT(start_year) AS has_start_year,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) AS pct_title,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    ROUND(try_divide(COUNT(lead_investigator), COUNT(*)) * 100.0, 1) AS pct_lead
FROM openalex.awards.moore_awards;


In [ ]:
%sql
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    ROUND(MIN(amount), 0) AS min_amount,
    ROUND(percentile_approx(amount, 0.5), 0) AS median_amount,
    ROUND(MAX(amount), 0) AS max_amount,
    collect_set(currency) AS currencies
FROM openalex.awards.moore_awards;


In [ ]:
%sql
SELECT funder_scheme, COUNT(*) AS rows,
       MIN(start_year) AS min_year, MAX(start_year) AS max_year,
       COUNT(amount) AS has_amount, ROUND(SUM(amount), 0) AS total_usd
FROM openalex.awards.moore_awards
GROUP BY funder_scheme
ORDER BY rows DESC;


In [ ]:
%sql
SELECT funder.id, funder.display_name, funder.ror_id, funder.doi, COUNT(*) AS rows
FROM openalex.awards.moore_awards
GROUP BY funder.id, funder.display_name, funder.ror_id, funder.doi;


In [ ]:
%sql
SELECT id, SUBSTRING(display_name, 1, 55) AS title,
       funder_scheme AS program, funding_type, start_year, amount, currency,
       lead_investigator.affiliation.name AS org
FROM openalex.awards.moore_awards
ORDER BY start_year DESC, amount DESC
LIMIT 10;
